# Huginn CCoT Finetuning
Finetunes three variants of Huginn on GSM8K and compares them against the frozen baseline.

| Experiment | `iter_injection` | `ccot_injection` | What trains |
|---|---|---|---|
| Baseline | `none` | `none` | nothing (frozen) |
| iter_only | `add` | `none` | `iter_proj` (~27M params) |
| ccot_only | `none` | `add` | `ccot_proj` (~27M params) |
| both | `add` | `add` | `iter_proj` + `ccot_proj` (~55M params) |

**`iter_injection`** — within a single query, injects the previous loop iteration's output back into the next iteration.  
**`ccot_injection`** — injects the final loop state from the *previous query* into the current query (cross-query episodic memory). Requires threading `ccot_memory` between queries during training.

**Recommended runtime:** A100 (40 GB). T4 (16 GB) works with gradient checkpointing enabled.

## 1. Setup

In [15]:
import subprocess, sys

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Mount Drive for saving checkpoints
from google.colab import drive
drive.mount('/content/drive')

NVIDIA A100-SXM4-40GB, 40960 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Clone repo (replace with your fork URL if you have one)
%cd /content/drive/MyDrive/recurrent-pretraining

# Install dependencies
!pip install transformers==4.47.1 accelerate datasets lm-eval -q

print("Setup complete.")

/content/drive/MyDrive/recurrent-pretraining
Setup complete.


## 2. Imports & helpers

In [19]:
import importlib.util, os, json, math, random, time
from pathlib import Path
from typing import Optional

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

# ── Bootstrap: bypass recpre/__init__.py (imports a removed torch 2.11 symbol) ──
sys.path.insert(0, '/content/recurrent-pretraining')
for _name, _path in [
    ('recpre.raven_config_minimal',   'recpre/raven_config_minimal.py'),
    ('recpre.raven_modeling_minimal', 'recpre/raven_modeling_minimal.py'),
]:
    _spec = importlib.util.spec_from_file_location(_name, _path)
    _mod  = importlib.util.module_from_spec(_spec)
    sys.modules[_name] = _mod
    _spec.loader.exec_module(_mod)

from recpre.raven_config_minimal import RavenConfig

DEVICE     = 'cuda'
DTYPE      = torch.bfloat16
MODEL_NAME = 'tomg-group-umd/huginn-0125'
SAVE_ROOT  = Path('/content/drive/MyDrive/huginn_ccot')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'torch {torch.__version__} | device: {DEVICE} | dtype: {DTYPE}')

torch 2.10.0+cu128 | device: cuda | dtype: torch.bfloat16


In [20]:
def load_model(iter_injection='none', ccot_injection='none', train_loop=True, gradient_checkpointing=False):
    """
    Load Huginn with the specified injection modes.
    Only the corresponding proj weights are trainable; everything else is frozen.
    """
    cfg = RavenConfig.from_pretrained(MODEL_NAME)
    cfg.iter_injection = iter_injection
    cfg.ccot_injection = ccot_injection

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        config=cfg,
        torch_dtype=DTYPE,
        device_map=DEVICE,
        ignore_mismatched_sizes=True,  # iter_proj / ccot_proj are new modules
    )

    if gradient_checkpointing:
        model.gradient_checkpointing_enable()

    # Freeze everything; unfreeze only the proj(s) we are training
    trainable_keywords = []
    if iter_injection != 'none':
        trainable_keywords.append('iter_proj')
    if ccot_injection != 'none':
        trainable_keywords.append('ccot_proj')

    if train_loop:
        trainable_keywords.append('core_block')  # unfreezes all 4 loop layers

    for name, param in model.named_parameters():
        param.requires_grad = any(kw in name for kw in trainable_keywords)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,} params  '
          f'({100*n_train/n_total:.2f}%)')
    print(f'  iter_injection={iter_injection!r}  ccot_injection={ccot_injection!r}')
    return model


def save_checkpoint(model, name: str):
    """Save full model state dict (includes the new proj weights)."""
    path = SAVE_ROOT / name
    model.save_pretrained(str(path))
    print(f'  Saved → {path}')


def load_checkpoint(name: str, iter_injection='none', ccot_injection='none'):
    """Load a saved checkpoint back into a model."""
    path = SAVE_ROOT / name
    cfg = RavenConfig.from_pretrained(str(path))
    cfg.iter_injection = iter_injection
    cfg.ccot_injection = ccot_injection
    model = AutoModelForCausalLM.from_pretrained(
        str(path), config=cfg, torch_dtype=DTYPE, device_map=DEVICE
    )
    model.eval()
    return model

## 3. Dataset — GSM8K

In [21]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_SEQ_LEN   = 256   # tokens per Q+A pair
WINDOW_SIZE   = 4     # consecutive problems threaded via ccot_memory

raw = load_dataset('openai/gsm8k', 'main')
print(f'Train: {len(raw["train"])}  |  Test: {len(raw["test"])}')


def format_example(q: str, a: str) -> str:
    # Strip the verbose chain-of-thought; keep only the final numeric answer
    final = a.split('####')[-1].strip()
    return f'Question: {q.strip()}\nAnswer: {final}'


def tokenize_example(q: str, a: str):
    """Returns input_ids and labels tensors (labels mask the question tokens)."""
    text   = format_example(q, a)
    q_text = f'Question: {q.strip()}\nAnswer:'

    full_ids = tokenizer(text,   add_special_tokens=True).input_ids
    q_ids    = tokenizer(q_text, add_special_tokens=True).input_ids

    full_ids = full_ids[:MAX_SEQ_LEN]
    labels   = [-100] * len(q_ids) + full_ids[len(q_ids):]
    labels   = labels[:MAX_SEQ_LEN]

    # Pad to MAX_SEQ_LEN
    pad_id = tokenizer.pad_token_id or 0
    pad_len = MAX_SEQ_LEN - len(full_ids)
    input_ids = full_ids + [pad_id] * pad_len
    labels    = labels   + [-100]   * pad_len

    return torch.tensor(input_ids), torch.tensor(labels)


class GSM8KDataset(Dataset):
    """Returns individual (input_ids, labels) pairs."""
    def __init__(self, split='train'):
        self.data = raw[split]

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        iids, labs = tokenize_example(row['question'], row['answer'])
        return {'input_ids': iids, 'labels': labs}


class GSM8KWindowDataset(Dataset):
    """
    Returns windows of WINDOW_SIZE consecutive problems for ccot_injection training.
    Each window is a list of (input_ids, labels) dicts.
    The model processes them in order, threading latent_states as ccot_memory.
    """
    def __init__(self, split='train', window_size=WINDOW_SIZE):
        self.data   = raw[split]
        self.wsize  = window_size
        self.starts = list(range(0, len(self.data) - window_size, window_size))

    def __len__(self):  return len(self.starts)

    def __getitem__(self, idx):
        start = self.starts[idx]
        window = []
        for i in range(start, start + self.wsize):
            row = self.data[i]
            iids, labs = tokenize_example(row['question'], row['answer'])
            window.append({'input_ids': iids, 'labels': labs})
        return window


def collate_single(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'labels':    torch.stack([b['labels']    for b in batch]),
    }

# Window collate: list of lists → list of batched dicts, one per step in window
def collate_window(batch):
    # batch = list of windows, each window = list of WINDOW_SIZE dicts
    window_size = len(batch[0])
    steps = []
    for step in range(window_size):
        steps.append({
            'input_ids': torch.stack([b[step]['input_ids'] for b in batch]),
            'labels':    torch.stack([b[step]['labels']    for b in batch]),
        })
    return steps  # list of WINDOW_SIZE batched dicts


print(f'Dataset ready. Window size: {WINDOW_SIZE}')

Train: 7473  |  Test: 1319
Dataset ready. Window size: 4


## 4. Training

In [22]:
# ── Hyperparameters ────────────────────────────────────────────────────────
LR            = 3e-4
EPOCHS        = 5
BATCH_SIZE    = 2     # windows (ccot) or examples (iter)
GRAD_ACCUM    = 16     # effective batch = BATCH_SIZE * GRAD_ACCUM
NUM_STEPS     = 32    # recurrent loop depth during training (< 32 to save memory)
BACKPROP_DEPTH = 8    # how many of NUM_STEPS carry gradients (TBPTT truncation)
WARMUP_RATIO  = 0.05
LOG_INTERVAL  = 20    # steps
GRAD_CLIP     = 1.0

# num_steps format for training: [no_grad_steps, grad_steps]
# iterate_forward runs the first slice detached, then the second with grad enabled.
NUM_STEPS_TRAIN = torch.tensor([NUM_STEPS - BACKPROP_DEPTH, BACKPROP_DEPTH])


def train(
    model,
    experiment_name: str,
    use_ccot_windows: bool,         # True for ccot_only / both; False for iter_only
    epochs: int = EPOCHS,
):
    """
    Generic training loop.
    - For iter_only: standard SFT dataloader, no ccot_memory threading.
    - For ccot_only / both: window dataloader, threads latent_states between
      consecutive problems as ccot_memory.
    """
    model.train()

    if use_ccot_windows:
        ds = GSM8KWindowDataset('train')
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                            collate_fn=collate_window, drop_last=True)
    else:
        ds = GSM8KDataset('train')
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                            collate_fn=collate_single, drop_last=True)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)

    total_steps    = len(loader) * epochs // GRAD_ACCUM
    warmup_steps   = max(1, int(total_steps * WARMUP_RATIO))
    scheduler      = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    global_step   = 0
    accum_loss    = 0.0
    accum_count   = 0

    output_details = {
        'return_logits': False, 'return_latents': True,
        'return_head': False,   'return_stats': False,
    }

    print(f'\n═══ Training: {experiment_name} ═══')
    print(f'  Epochs={epochs}  Steps/epoch={len(loader)}  '
          f'Effective batch={BATCH_SIZE*GRAD_ACCUM}  LR={LR}')
    print(f'  num_steps={NUM_STEPS}  backprop_depth={BACKPROP_DEPTH}  '
          f'(no-grad={NUM_STEPS - BACKPROP_DEPTH}, grad={BACKPROP_DEPTH})')

    for epoch in range(epochs):
        for batch_idx, batch in enumerate(loader):

            # ── Standard SFT (iter_only) ────────────────────────────────
            if not use_ccot_windows:
                iids   = batch['input_ids'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)

                out = model(
                    input_ids=iids,
                    labels=labels,
                    num_steps=NUM_STEPS_TRAIN,
                    output_details=output_details,
                )
                loss = out.loss / GRAD_ACCUM

            # ── Window SFT with ccot_memory threading ───────────────────
            else:
                # batch = list of WINDOW_SIZE batched dicts
                ccot_memory = None
                window_loss = torch.tensor(0.0, device=DEVICE)

                for step_batch in batch:
                    iids   = step_batch['input_ids'].to(DEVICE)
                    labels = step_batch['labels'].to(DEVICE)

                    out = model(
                        input_ids=iids,
                        labels=labels,
                        num_steps=NUM_STEPS_TRAIN,
                        ccot_memory=ccot_memory,
                        output_details=output_details,
                    )
                    window_loss  = window_loss + out.loss
                    # Detach: no BPTT across query boundaries
                    ccot_memory  = out.latent_states.detach()

                loss = (window_loss / len(batch)) / GRAD_ACCUM

            loss.backward()
            accum_loss  += loss.item() * GRAD_ACCUM
            accum_count += 1

            if (batch_idx + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % LOG_INTERVAL == 0:
                    avg = accum_loss / accum_count
                    lr  = scheduler.get_last_lr()[0]
                    print(f'  epoch={epoch+1}  step={global_step}  '
                          f'loss={avg:.4f}  lr={lr:.2e}')
                    accum_loss  = 0.0
                    accum_count = 0

        print(f'  ── Epoch {epoch+1} complete ──')

    model.eval()
    save_checkpoint(model, experiment_name)
    return model


print('Training helpers ready.')

Training helpers ready.


## 5. Experiment 1 — `iter_only`
Trains `iter_proj` only. Within each query, iteration t injects the output of iteration t-1 back into the loop.

In [23]:
print('Loading model for iter_only...')
model_iter = load_model(iter_injection='add', ccot_injection='none')

model_iter = train(
    model_iter,
    experiment_name='iter_only',
    use_ccot_windows=False,  # standard SFT — no memory threading needed
)

del model_iter
torch.cuda.empty_cache()
print('iter_only done.')

Loading model for iter_only...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of RavenForCausalLM were not initialized from the model checkpoint at tomg-group-umd/huginn-0125 and are newly initialized: ['transformer.ccot_proj.weight', 'transformer.iter_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Trainable: 27,878,400 / 3,620,733,600 params  (0.77%)
  iter_injection='add'  ccot_injection='none'

═══ Training: iter_only ═══
  Epochs=2  Steps/epoch=1868  Effective batch=16  LR=0.0003
  num_steps=16  backprop_depth=4  (no-grad=12, grad=4)
  epoch=1  step=20  loss=1.5072  lr=1.30e-04
  epoch=1  step=40  loss=0.5574  lr=2.61e-04
  epoch=1  step=60  loss=0.8811  lr=3.00e-04
  epoch=1  step=80  loss=0.7410  lr=2.99e-04
  epoch=1  step=100  loss=1.3555  lr=2.97e-04
  epoch=1  step=120  loss=0.8009  lr=2.95e-04
  epoch=1  step=140  loss=0.7446  lr=2.92e-04
  epoch=1  step=160  loss=0.7748  lr=2.88e-04
  epoch=1  step=180  loss=0.7543  lr=2.83e-04
  epoch=1  step=200  loss=0.8750  lr=2.78e-04
  epoch=1  step=220  loss=0.7187  lr=2.72e-04
  epoch=1  step=240  loss=0.5553  lr=2.66e-04
  epoch=1  step=260  loss=0.5974  lr=2.59e-04
  epoch=1  step=280  loss=0.5645  lr=2.51e-04
  epoch=1  step=300  loss=0.4245  lr=2.43e-04
  epoch=1  step=320  loss=0.6459  lr=2.35e-04
  epoch=1  step=340  l

## 6. Experiment 2 — `ccot_only`
Trains `ccot_proj` only. Consecutive problems in each window are threaded via `ccot_memory` so the model learns to use the previous query's latent state.

In [24]:
print('Loading model for ccot_only...')
model_ccot = load_model(iter_injection='none', ccot_injection='add')

model_ccot = train(
    model_ccot,
    experiment_name='ccot_only',
    use_ccot_windows=True,   # thread latent_states across consecutive problems
)

del model_ccot
torch.cuda.empty_cache()
print('ccot_only done.')

Loading model for ccot_only...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of RavenForCausalLM were not initialized from the model checkpoint at tomg-group-umd/huginn-0125 and are newly initialized: ['transformer.ccot_proj.weight', 'transformer.iter_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Trainable: 27,878,400 / 3,620,733,600 params  (0.77%)
  iter_injection='none'  ccot_injection='add'

═══ Training: ccot_only ═══
  Epochs=2  Steps/epoch=467  Effective batch=16  LR=0.0003
  num_steps=16  backprop_depth=4  (no-grad=12, grad=4)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

## 7. Experiment 3 — `both`
Trains both `iter_proj` and `ccot_proj` simultaneously.

In [ ]:
print('Loading model for both...')
model_both = load_model(iter_injection='add', ccot_injection='add')

model_both = train(
    model_both,
    experiment_name='both',
    use_ccot_windows=True,   # window dataloader so ccot can be threaded
)

del model_both
torch.cuda.empty_cache()
print('both done.')

## 8. Evaluation
Compare all four models (baseline + 3 finetuned) on held-out GSM8K test examples.

Two metrics:
- **Exact match** — final numeric answer matches the gold label
- **Depth sweep** — accuracy at steps ∈ {4, 8, 16, 32} to see if finetuning changed the optimal recurrence depth

In [25]:
import re

N_EVAL       = 200   # test examples to evaluate
EVAL_STEPS   = [4, 8, 16, 32]


def extract_number(text: str) -> Optional[str]:
    """Pull the last number from a string (handles commas, decimals)."""
    nums = re.findall(r'-?\d[\d,]*\.?\d*', text.replace(',', ''))
    return nums[-1] if nums else None


@torch.no_grad()
def evaluate_model(
    model,
    n: int = N_EVAL,
    num_steps_list=EVAL_STEPS,
    use_ccot: bool = False,
):
    """
    Returns a dict: {num_steps: accuracy}.
    If use_ccot=True, threads latent_states between consecutive test examples
    (simulating what the model was trained to do).
    """
    test_data = raw['test'].select(range(n))
    results   = {s: {'correct': 0, 'total': 0} for s in num_steps_list}

    for num_steps in num_steps_list:
        ccot_memory = None

        for row in test_data:
            gold = extract_number(row['answer'].split('####')[-1])
            prompt = f"Question: {row['question'].strip()}\nAnswer:"

            enc    = tokenizer(prompt, return_tensors='pt')
            iids   = enc['input_ids'].to(DEVICE)
            amask  = enc['attention_mask'].to(DEVICE)

            # Generate — num_steps as plain int for inference
            out = model.generate(
                input_ids=iids,
                attention_mask=amask,
                max_new_tokens=32,
                num_steps=num_steps,
                do_sample=False,
            )
            generated = tokenizer.decode(
                out[0][iids.shape[1]:], skip_special_tokens=True
            )
            pred = extract_number(generated)

            results[num_steps]['correct'] += int(pred == gold)
            results[num_steps]['total']   += 1

            # Thread ccot_memory for ccot-enabled models
            if use_ccot:
                fwd = model(
                    input_ids=iids,
                    num_steps=num_steps,   # plain int for eval (no grad tracking needed)
                    ccot_memory=ccot_memory,
                    output_details={'return_logits': False, 'return_latents': True,
                                    'return_head': False, 'return_stats': False},
                )
                ccot_memory = fwd.latent_states.detach()

    return {s: r['correct'] / r['total'] for s, r in results.items()}


print('Evaluation helpers ready.')

Evaluation helpers ready.


In [ ]:
# ── Load all four models for evaluation ────────────────────────────────────
experiments = [
    # (label,       iter_inj, ccot_inj, checkpoint_name, use_ccot_eval)
    ('Baseline',    'none',   'none',   None,          False),
    ('iter_only',   'add',    'none',   'iter_only',   False),
    #('ccot_only',   'none',   'add',    'ccot_only',   True),
    #('both',        'add',    'add',    'both',        True),
]

all_results = {}

for label, iter_inj, ccot_inj, ckpt_name, use_ccot_eval in experiments:
    print(f'\nEvaluating: {label}')

    if ckpt_name is None:
        # Baseline: load frozen pretrained model
        cfg = RavenConfig.from_pretrained(MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, config=cfg, torch_dtype=DTYPE,
            device_map=DEVICE, ignore_mismatched_sizes=True
        )
        model.eval()
    else:
        model = load_checkpoint(ckpt_name, iter_inj, ccot_inj)

    res = evaluate_model(model, use_ccot=use_ccot_eval)
    all_results[label] = res

    for steps, acc in res.items():
        print(f'  steps={steps:2d}  acc={acc*100:.1f}%')

    del model
    torch.cuda.empty_cache()

print('\nAll evaluations complete.')


Evaluating: Baseline


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of RavenForCausalLM were not initialized from the model checkpoint at tomg-group-umd/huginn-0125 and are newly initialized: ['transformer.ccot_proj.weight', 'transformer.iter_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 9. Results table

In [ ]:
import pandas as pd

rows = []
for label, res in all_results.items():
    row = {'Model': label}
    row.update({f'steps={s}': f'{v*100:.1f}%' for s, v in res.items()})
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df.to_csv(str(SAVE_ROOT / 'results.csv'))
print(f'\nSaved → {SAVE_ROOT}/results.csv')

In [ ]:
# ── Plot accuracy vs recurrence depth ─────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
styles = {'Baseline': ('black', '--'), 'iter_only': ('royalblue', '-'),
          'ccot_only': ('tomato', '-'), 'both': ('seagreen', '-')}

for label, res in all_results.items():
    xs = sorted(res.keys())
    ys = [res[x] * 100 for x in xs]
    color, ls = styles.get(label, ('gray', '-'))
    ax.plot(xs, ys, marker='o', label=label, color=color, linestyle=ls)

ax.set_xlabel('num_steps (recurrence depth)')
ax.set_ylabel('Accuracy (%) on GSM8K test')
ax.set_title('Huginn CCoT — Accuracy vs Recurrence Depth')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(str(SAVE_ROOT / 'accuracy_vs_steps.png'), dpi=150)
plt.show()

## 10. Qualitative comparison
Run the same prompts through all four models and print outputs side-by-side.

In [ ]:
QUAL_PROMPTS = [
    "Q: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?\nA:",
    "Q: Sarah has 24 apples. She gives half to Tom, then buys 6 more. How many does she have?\nA:",
    "Q: If it takes 5 machines 5 minutes to make 5 widgets, how long does 1 machine take to make 1 widget?\nA:",
]

@torch.no_grad()
def quick_generate(model, prompt, num_steps=16, ccot_memory=None):
    enc   = tokenizer(prompt, return_tensors='pt')
    iids  = enc['input_ids'].to(DEVICE)
    amask = enc['attention_mask'].to(DEVICE)
    out   = model.generate(input_ids=iids, attention_mask=amask,
                           max_new_tokens=40, num_steps=num_steps, do_sample=False)
    return tokenizer.decode(out[0][iids.shape[1]:], skip_special_tokens=True).strip()


for prompt in QUAL_PROMPTS:
    print(f'PROMPT: {prompt[:80]}...')
    print('-' * 70)

    for label, iter_inj, ccot_inj, ckpt_name, _ in experiments:
        if ckpt_name is None:
            cfg = RavenConfig.from_pretrained(MODEL_NAME)
            m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, config=cfg,
                    torch_dtype=DTYPE, device_map=DEVICE, ignore_mismatched_sizes=True)
        else:
            m = load_checkpoint(ckpt_name, iter_inj, ccot_inj)
        m.eval()

        ans = quick_generate(m, prompt, num_steps=16)
        print(f'  [{label:<12}] {ans[:80]}')

        del m; torch.cuda.empty_cache()

    print()